In [4]:
from tqdm.notebook import tqdm
from transformers import pipeline

In [5]:
classifier = pipeline("sentiment-analysis")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 617.11it/s, Materializing param=pre_classifier.weight]                                  


In [31]:
classifier(
    [
    "My name is Rokas. I'm a data engineer from Lithuania.",
    "My hometown is Šiauliai"
    ]
)

[{'label': 'POSITIVE', 'score': 0.9764316082000732},
 {'label': 'POSITIVE', 'score': 0.9897035956382751}]

## Tokenizer

In [7]:
from transformers import AutoTokenizer

In [8]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [9]:
tokenizer

BertTokenizer(name_or_path='distilbert-base-uncased-finetuned-sst-2-english', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [11]:
raw_inputs = [
    "My name is Rokas. I'm a data engineer from Lithuania.",
    "My hometown is Šiauliai"
]

inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")

In [12]:
inputs

{'input_ids': tensor([[  101,  2026,  2171,  2003, 20996, 13716,  1012,  1045,  1005,  1049,
          1037,  2951,  3992,  2013,  9838,  1012,   102],
        [  101,  2026,  9627,  2003,  9033,  4887,  6632,  2072,   102,     0,
             0,     0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}

## Model

### For hidden state retrieval
<!-- Add image from assets/hidden.png here -->
![alt](assets/hidden.png)

In [13]:
from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 798.95it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 17, 768])


In [23]:
outputs

BaseModelOutput(last_hidden_state=tensor([[[ 0.6308,  0.6842,  0.1941,  ..., -0.2722,  0.1274,  0.2014],
         [ 0.8178,  0.6770, -0.4905,  ..., -0.0017,  0.4401,  0.2687],
         [ 0.4040,  0.8812, -0.3283,  ...,  0.0882,  0.5429, -0.6079],
         ...,
         [ 0.5940,  0.5009,  0.0863,  ..., -0.5871, -0.5921, -0.4820],
         [ 0.5744,  0.0506,  0.0419,  ...,  0.3396, -0.1624, -1.0273],
         [ 0.6562,  0.1136,  0.1754,  ...,  0.3899, -0.2636, -0.5570]],

        [[ 0.6706,  0.0427,  0.5982,  ...,  0.0063,  0.2835, -0.2234],
         [ 0.9916,  0.1047, -0.0628,  ..., -0.1955, -0.7886, -0.2316],
         [ 0.3160,  0.0160,  0.1008,  ..., -0.1835,  0.6244, -0.5683],
         ...,
         [ 0.7660, -0.5216,  0.3359,  ...,  0.1732,  0.1892, -0.3189],
         [ 0.7756, -0.6978,  0.2491,  ...,  0.4050,  0.0680, -0.3627],
         [ 0.4741, -0.4924,  0.5383,  ...,  0.1601, -0.1032, -0.2548]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)

In [22]:
outputs[0][1][15]

tensor([ 7.7559e-01, -6.9780e-01,  2.4913e-01,  6.6923e-02,  1.5229e-01,
         1.3127e-01,  3.8837e-01,  1.0672e+00, -3.4261e-01, -4.9542e-03,
         8.5871e-03, -4.0142e-01, -5.0814e-01, -3.1274e-01,  2.4439e-01,
         7.3025e-01,  8.0711e-01,  7.7047e-01,  7.1139e-01, -5.2395e-01,
         2.5995e-01,  1.7204e-01,  3.0237e-01,  1.1192e+00,  6.0464e-02,
        -1.9230e-01, -1.2525e-01,  2.6484e-01, -1.5712e-01, -3.1689e-02,
        -2.9936e-01,  6.3293e-02, -1.5714e-01,  7.2262e-01,  4.8977e-01,
         3.6627e-02, -4.2258e-01, -3.7375e-02, -7.4302e-02, -3.0393e-01,
        -2.4972e-02, -5.7683e-01,  1.2199e+00, -1.2080e+00, -2.3735e-01,
        -7.4209e-01,  1.0578e+00,  4.4285e-01, -4.1847e-01,  4.6350e-01,
        -8.8999e-02, -3.0013e-01, -3.4699e-01,  8.3155e-01,  2.9668e-01,
         6.8991e-01, -5.8624e-01,  1.7187e-01, -2.8069e-01, -2.9528e-01,
         1.1305e-01,  2.2752e-01,  1.4521e-01,  1.5578e-02,  1.0337e+00,
         4.9608e-01,  3.8423e-01,  8.1970e-02, -2.4

## Model for classification

![alt](assets/whole.png)

In [24]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 641.09it/s, Materializing param=pre_classifier.weight]                                  


In [25]:
outputs = model(**inputs)

In [26]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-1.8254,  1.8986],
        [-2.2712,  2.2944]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [27]:
print(outputs.logits.shape)

torch.Size([2, 2])


### Aplly softmax

In [28]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)


In [29]:
print(predictions)

tensor([[0.0236, 0.9764],
        [0.0103, 0.9897]], grad_fn=<SoftmaxBackward0>)


In [30]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}